## 1D Correlation

In [3]:
%%writefile q8.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define N 60
#define LOW 10
#define HIGH 99
#define BLOCK_SIZE 8
#define MASK_WIDTH 5

// ── CUDA Kernel
__global__ void corr1DKernel(int *dN, int *dM, int *dP, int maskWidth, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        int dPvalue = 0;
        for (int j = 0; j < maskWidth; j++) {
            if ((i + j) < n)
                dPvalue += dN[i + j] * dM[j];
        }
        dP[i] = dPvalue;
    }
}

// ── Host Function
__host__ void corr1D(int *hN, int *hM, int *hP, int maskWidth, int n)
{
    int *dN, *dM, *dP;
    int sizeN = n * sizeof(int);
    int sizeM = maskWidth * sizeof(int);

    cudaMalloc((void**)&dN, sizeN);
    cudaMalloc((void**)&dM, sizeM);
    cudaMalloc((void**)&dP, sizeN);

    cudaMemcpy(dN, hN, sizeN, cudaMemcpyHostToDevice);
    cudaMemcpy(dM, hM, sizeM, cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_SIZE, 1, 1);
    dim3 DimGrid((int)ceil((float)n / BLOCK_SIZE), 1, 1);

    corr1DKernel<<<DimGrid, DimBlock>>>(dN, dM, dP, maskWidth, n);

    cudaMemcpy(hP, dP, sizeN, cudaMemcpyDeviceToHost);

    cudaFree(dN); cudaFree(dM); cudaFree(dP);
}

// ── Main
int main()
{
    int hN[N], hM[MASK_WIDTH], hP[N];

    srand(time(NULL));
    for (int i = 0; i < N; i++)
        hN[i] = rand() % (HIGH - LOW + 1) + LOW;

    // Fixed mask
    int mask[MASK_WIDTH] = {3, 4, 5, 4, 3};
    for (int i = 0; i < MASK_WIDTH; i++)
        hM[i] = mask[i];

    printf("hN:\n");
    for (int i = 0; i < N; i++) printf("%4d", hN[i]);
    printf("\n");

    printf("\nhM (Mask):\n");
    for (int i = 0; i < MASK_WIDTH; i++) printf("%4d", hM[i]);
    printf("\n");

    corr1D(hN, hM, hP, MASK_WIDTH, N);

    printf("\nhP (1D Correlation Result):\n");
    for (int i = 0; i < N; i++) printf("%6d", hP[i]);
    printf("\n");

    return 0;
}

Overwriting q8.cu


In [4]:
!nvcc -arch=sm_75 q8.cu -o q8

!./q8

hN:
  32  89  19  85  99  63  13  50  60  12  44  56  83  54  64  89  74  80  72  53  20  77  37  69  75  69  13  40  37  68  95  21  57  66  96  56  30  62  96  80  64  92  36  99  47  52  88  73  32  23  26  94  90  53  26  28  75  29  58  12

hM (Mask):
   3   4   5   4   3

hP (1D Correlation Result):
  1184  1353  1183  1168   994   767   719   794   921   986  1179  1293  1363  1399  1454  1410  1174  1104   947   979  1054  1231  1077  1024   837   811   941  1051  1113  1115  1206  1173  1229  1168  1198  1222  1330  1502  1404  1397  1277  1259  1211  1316  1177  1070   886   848   964  1162  1194  1095   949   790   855   797   679   379   222    36
